# Protocol B model and fold training
Attach MSLD v2.0 and the frozen Protocol B training inputs. Set MODEL and FOLD in the job cell. To resume, attach its full saved output and select RESUME=True.


In [ ]:
# Install the small pinned dependencies; retain Kaggle's GPU-enabled torch/torchvision.
import importlib.metadata as metadata
import subprocess
import sys
required = {'numpy':'2.1.3','pandas':'2.3.3','scipy':'1.16.3',
            'scikit-learn':'1.6.1','timm':'1.0.29','Pillow':'13.0.0',
            'huggingface_hub':'1.33.0','safetensors':'0.8.0'}
changes=[]
for package,version in required.items():
    try: installed=metadata.version(package)
    except metadata.PackageNotFoundError: installed=None
    if installed!=version:changes.append(f'{package}=={version}')
if changes:
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps',*changes],check=True)
print('Setup ready. Kaggle torch and torchvision have been retained.')


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import stat
from zipfile import ZipFile
EXPECTED_MANIFEST_SHA = '856d1724988275704854984023a5c9be4d39869fb02187bcd011118c0de20200'
INPUT_ROOT=Path('/kaggle/input')
CODE_DIR=Path('/kaggle/working/pb_corrected_code')
def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
markers=list(INPUT_ROOT.rglob('pb_corrected_manifest.json'))
if not markers:
    for archive in INPUT_ROOT.rglob('PB_Corrected_Inputs.zip'):
        target=Path('/tmp/pb_corrected_bootstrap')/_sha(archive)
        with ZipFile(archive) as z:
            if sum(i.file_size for i in z.infolist())>100*1024*1024:
                raise ValueError('Unexpectedly large input package.')
            for info in z.infolist():
                rel=PurePosixPath(info.filename.replace('\\','/'))
                if rel.is_absolute() or '..' in rel.parts or any(':' in p for p in rel.parts) or stat.S_ISLNK(info.external_attr >> 16):
                    raise ValueError('Unsafe archive path.')
                dest=target.joinpath(*rel.parts)
                if info.is_dir():dest.mkdir(parents=True,exist_ok=True)
                else:
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(info) as source,dest.open('wb') as output:shutil.copyfileobj(source,output)
        markers.extend(target.rglob('pb_corrected_manifest.json'))
markers=sorted(p for p in markers if _sha(p)==EXPECTED_MANIFEST_SHA)
if not markers:
    raise FileNotFoundError('Attach the NEW PB_Corrected_Inputs.zip as a Kaggle dataset. Old completion/final-run inputs will not work.')
INPUTS=markers[0].parent
frozen=json.loads(markers[0].read_text())
CODE_DIR.mkdir(parents=True,exist_ok=True)
for name in ['corrected_core.py','resume_engine.py','train_fold.py','run_jobs.py','merge_corrected.py']:
    source=INPUTS/'code'/name
    if _sha(source)!=frozen['files']['code/'+name]['sha256']:raise ValueError('Packaged code changed.')
    shutil.copyfile(source,CODE_DIR/name)
if str(CODE_DIR) not in sys.path:sys.path.insert(0,str(CODE_DIR))
print('Corrected input package and code verified.')


In [ ]:
MODEL='swin_small'
FOLD=1
RESUME=False
SESSION_HOURS=4.5
command=[sys.executable,str(CODE_DIR/'run_jobs.py'),'--jobs',f'{MODEL}:{FOLD}',
         '--expected-manifest-sha',EXPECTED_MANIFEST_SHA,'--input-root',str(INPUT_ROOT),
         '--max-hours',str(SESSION_HOURS)]
if RESUME:command.append('--require-resume')
subprocess.run(command,check=True)
